In [1]:
import io
import json
import logging
import os
import uuid
from io import BytesIO
from typing import Any

import oci.ai_document
import oci.object_storage
from datasets import load_dataset
from oci.config import from_file
from oci.generative_ai_inference import GenerativeAiInferenceClient
from oci.generative_ai_inference.models import (ChatDetails, ImageContent,
                                                TextContent)

/Users/valefeve/Projects/oci-multimodal-genai-vs-odu/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset("nikinbaidar/handwritten")

In [3]:
config = from_file()

In [4]:
try:
    object_storage_client = oci.object_storage.ObjectStorageClient(config=config)
    ai_document_client = oci.ai_document.AIServiceDocumentClientCompositeOperations(oci.ai_document.AIServiceDocumentClient(config))

    if os.getenv("COMPARTMENT_OCID") is not None:
        compartment_ocid = os.getenv('COMPARTMENT_OCID')
    else:
        raise ValueError("ERROR: Missing configuration key  COMPARTMENT_OCID ")

    if os.getenv("NAMESPACE") is not None:
        namespace = os.getenv('NAMESPACE')
    else:
        raise ValueError("ERROR: Missing configuration key  NAMESPACE_NAME ")

    if os.getenv("BUCKET_NAME") is not None:
        bucket_name = os.getenv('BUCKET_NAME')
    else:
        raise ValueError("ERROR: Missing configuration key  BUCKETNAME ")

    if os.getenv("MODEL_ID") is not None:
        model_id = os.getenv('MODEL_ID')
    else:
        raise ValueError("ERROR: Missing configuration key  MODEL_ID ")

    input_location = oci.ai_document.models.ObjectLocation()
    input_location.namespace_name = namespace
    input_location.bucket_name = bucket_name

    # Setup the output location where processor job results will be created
    output_location = oci.ai_document.models.OutputLocation()
    output_location.namespace_name = namespace
    output_location.bucket_name = bucket_name
    output_location.prefix = "ocr"

except Exception as e:
   logging.error("Error during client configuration: " + str(e))
   raise e

In [5]:
output_location.namespace_name

'idi1o0a010nx'

In [6]:
def upload_file_to_object_storage(bucket_name: str, file_path: str) -> str:
    try:
        object_name = os.path.basename(file_path)
        with open(file_path, 'rb') as file:
            response = object_storage_client.put_object(
                namespace_name=namespace,
                bucket_name=bucket_name,
                object_name=object_name,
                put_object_body=file,
                content_type="image/png"
            )
            return response
    except Exception as e:
        logging.error("Error uploading file to Object Storage: " + str(e))
        raise e

def upload_object_to_object_storage(bucket_name: str, object_name: str, data: bytes) -> str:
    try:
        response = object_storage_client.put_object(
            namespace_name=namespace,
            bucket_name=bucket_name,
            object_name=object_name,
            put_object_body=data
        )
        return response
    except Exception as e:
        logging.error("Error uploading object to Object Storage: " + str(e))
        raise e
        

In [7]:
image = ds['train'][0]['image']
buf = BytesIO()
image.save(buf, format="PNG")
png_bytes = buf.getvalue()
response = upload_object_to_object_storage(bucket_name, "test_image3.png", png_bytes)
response.status

200

In [8]:
def get_file_from_object_storage(bucket_name: str, object_name: str) -> Any:
    try:
        response = object_storage_client.get_object(
            namespace_name=namespace,
            bucket_name=bucket_name,
            object_name=object_name
        )
        return response.data.content
    except Exception as e:
        logging.error("Error retrieving file from Object Storage: " + str(e))
        raise e

In [9]:
aiservicedocument_client = oci.ai_document.AIServiceDocumentClientCompositeOperations(
            oci.ai_document.AIServiceDocumentClient(config=config)
        )

In [10]:
def create_processor_job_callback(self, times_called: int, response: Any) -> None:
    print("Waiting for processor lifecycle state to go into succeeded state:", response.data)


In [11]:
def extract_text_from_document(document_file_name: str):
    try:
        text_extraction_feature = oci.ai_document.models.DocumentTextExtractionFeature()

        input_location = oci.ai_document.models.ObjectStorageLocations(
            object_locations=[oci.ai_document.models.ObjectLocation(namespace_name=namespace,
                                                                     bucket_name=bucket_name,
                                                                     object_name=document_file_name)]
        )

        output_loc = oci.ai_document.models.OutputLocation(
            namespace_name=output_location.namespace_name,
            bucket_name=output_location.bucket_name,
            prefix=output_location.prefix
        )

        create_processor_job_details_text_extraction = oci.ai_document.models.CreateProcessorJobDetails(
            display_name=str(uuid.uuid4()),
            compartment_id=compartment_ocid,
            input_location=input_location,
            output_location=output_loc,
            processor_config=oci.ai_document.models.GeneralProcessorConfig(features=[text_extraction_feature])
        )

        # print("Calling create_processor with create_processor_job_details_text_extraction:", create_processor_job_details_text_extraction)

        create_processor_response = ai_document_client.create_processor_job_and_wait_for_state(
            create_processor_job_details=create_processor_job_details_text_extraction,
            wait_for_states=[oci.ai_document.models.ProcessorJob.LIFECYCLE_STATE_SUCCEEDED],
            waiter_kwargs={"wait_callback": create_processor_job_callback}
        )

        print("processor call succeeded with status: {} and request_id: {}.".format(create_processor_response.status, create_processor_response.request_id))

        result = create_processor_response.data

        return create_processor_response

    except Exception as e:
        logging.error("Error during document processing: " + str(e))
        raise e

In [12]:
SAMPLE_INVOICES_PATH="sample_invoices/"

In [13]:
jobs = []

cont = 1
for item in ds['test']:
    image = item['image']
    buf = BytesIO()
    image.save(buf, format="PNG")
    png_bytes = buf.getvalue()
    response = upload_object_to_object_storage(bucket_name, item['id'], png_bytes)
    if response.status == 200:
        print(f"Uploaded {item['id']} successfully.")
    else:
        print(f"Failed to upload {item['id']}. Status code: {response.status}")
        continue
    ocr_response = extract_text_from_document(item['id'])
    print(f"OCR Response for {item['id']}: {ocr_response}")
    jobs.append({
        "job_id": ocr_response.data.id,
        "file_name": item['id']
    })
    cont += 1
    if cont > 5:
        break

Uploaded eng_EU_307.jpg successfully.
processor call succeeded with status: 200 and request_id: BAC96BF70A584EA88C5B1C3FDED87270/A3A3A405C7062844C21E7A694F524411/6E253A04A998A1AA0E7D162C405BF97B.
OCR Response for eng_EU_307.jpg: <oci.response.Response object at 0x1674d20d0>
Uploaded eng_EU_184.jpg successfully.
processor call succeeded with status: 200 and request_id: 7D88968E45C44CB588FF5F790B3EF345/27F9735FCBC77B9DC71E7A694F524453/73A73438F5767EF59976A00D2A36B9A6.
OCR Response for eng_EU_184.jpg: <oci.response.Response object at 0x1674f1e90>
Uploaded eng_NA_034.jpg successfully.
processor call succeeded with status: 200 and request_id: D96DB70C94164861912E225D0B221B43/D30E8476E77C2338CA1E7A694F524470/A6C557F27EBBC88FF0ACDFC792DD5295.
OCR Response for eng_NA_034.jpg: <oci.response.Response object at 0x1674efc90>
Uploaded eng_AS_016.jpg successfully.
processor call succeeded with status: 200 and request_id: D976BA9547C0466390A0E790EBC99A8D/DA8526B53B711DD0CD1E7A694F524472/B37E679EC32FD

In [14]:
results_path = f"{output_location.prefix}/{create_processsor_job_response.data.id}/{namespace}_{bucket_name}/results/image1.pdf.json"
result = get_file_from_object_storage(bucket_name, results_path)
json.loads(result)
text = ""
for page in json.loads(result)["pages"]:
    for line in page["lines"]:
        print(line["text"])
        text += line["text"] + "\n"

NameError: name 'create_processsor_job_response' is not defined

In [20]:

response = upload_file_to_object_storage(bucket_name, "./" + SAMPLE_INVOICES_PATH + "image2.pdf")
create_processsor_job_response = extract_text_from_document("image2.pdf")

results_path = f"{output_location.prefix}/{create_processsor_job_response.data.id}/{namespace}_{bucket_name}/results/image2.pdf.json"
result = get_file_from_object_storage(bucket_name, results_path)
json.loads(result)
text = ""
for page in json.loads(result)["pages"]:
    for line in page["lines"]:
        print(line["text"])
        text += line["text"] + "\n"
        

processor call succeeded with status: 200 and request_id: 72B6F05845BB4E10BFB498B5DCB2AB52/3478500555CDEBF3BB1F7A694F524406/C366A5FE9835308CB7E59B09CEDF726D.
INVERSIONES PURNATA S.A.C.
Calle alcanfores 373 Miraflores
Barra Maretazo
RUC: 20613695487
Tel: 01-2378385 /
Email: ggarridos@gmail.com
Operacisn: 01010125
PRE-CUENTA
MOZO: Milko
NO FISCAL / NO FISCAL
NESA: M3
CANT. PERSONAS: 2
CLIENTE: CLIENTE GENWRICO
DOC: -
01/01/2026 20:55:23
Articulo
Cant P.U Importe
Arroz Con Pollo
1
42.00
42.00
(Combo) CEV MIX +
1
47.00
47.00
CHICHA PES
Chicha - Vaso
1
8.00
8.00
Maracuya - Vaso
I
8.00
8.00
Ronda Marina (1 Unidad)
2
0.00
0.00
Agua San Luis C/g
1
8.00
8.00
Total Consumo:
S/ 113.00
Total a pagar:
S/ 113.00
RUC:
RAZON SOCIAL:
-
-
--
--
DIRECCION:
GRACIAS


In [21]:
generative_ai_client = GenerativeAiInferenceClient(config)

In [22]:
# === Prompt del usuario ===
user_input = """
For the text extracted from the document, generate a json file with the following fields: \
RUC: <Supplier's RUC> # Corresponds to an 11-digit number. \
Company Name: <Supplier's Company Name> \
Address: <Supplier's Address> \
Invoice Number: <Invoice Number> \
Invoice Date: <Invoice Date in format DD/MM/YYYY> \
Total Amount: <Total Amount in format 0.00> \
Currency: <Currency Code, e.g., PEN or USD> \
Tax Amount: <Tax Amount in format 0.00> \
Payment Terms: <Payment Terms, e.g., "Contado" or "30 days">

Additionaly, list the items in the invoice with the following fields for each item: \
Items: \
    - Description: <Item Description> \
    - Quantity: <Item Quantity in format 0.00> \
    - Unit Price: <Item Unit Price in format 0.00> \
    - Total Price: <Item Total Price in format 0.00>

Example of the expected JSON format:
{
  "RUC": "12345678901",
  "Company Name": "ABC S.A.C.",
  "Address": "Av. Example 123, Lima, Peru",
  "Invoice Number": "F001-00012345",
  "Invoice Date": "15/08/2023",
  "Total Amount": "1500.00",
  "Currency": "PEN",
  "Tax Amount": "270.00",
  "Payment Terms": "30 days",
  "Items": [
    {
      "Description": "Product A",
      "Quantity": "2.00",
      "Unit Price": "500.00",
      "Total Price": "1000.00"
    },
    {
      "Description": "Product B",
      "Quantity": "1.00",
      "Unit Price": "500.00",
      "Total Price": "500.00"
    }
  ]
}
    
Please ensure the JSON is properly formatted.

Here is the extracted text from the document:
"""

# --- Construcción del request ---
content = oci.generative_ai_inference.models.TextContent(text=user_input+"\n\n"+text)
message = oci.generative_ai_inference.models.Message(role="USER", content=[content])

chat_request = oci.generative_ai_inference.models.GenericChatRequest(
    api_format=oci.generative_ai_inference.models.BaseChatRequest.API_FORMAT_GENERIC,
    messages=[message],
    max_tokens=600,
    temperature=1.0,
    frequency_penalty=0.0,
    presence_penalty=0.0,
    top_p=0.75,
)

chat_details = oci.generative_ai_inference.models.ChatDetails(
    serving_mode=oci.generative_ai_inference.models.OnDemandServingMode(model_id=model_id),
    chat_request=chat_request,
    compartment_id=compartment_ocid,
)

In [23]:
response = generative_ai_client.chat(
    chat_details=chat_details
)

In [24]:
print(response.data.chat_response.choices[0].message.content[0].text)

Based on the provided extracted text, I will generate a JSON file with the required fields. Here is the resulting JSON:

```json
{
  "RUC": "20613695487",
  "Company Name": "INVERSIONES PURNATA S.A.C.",
  "Address": "Calle alcanfores 373 Miraflores",
  "Invoice Number": "PRE-CUENTA",
  "Invoice Date": "01/01/2026",
  "Total Amount": "113.00",
  "Currency": "PEN",
  "Tax Amount": "0.00",
  "Payment Terms": "Contado",
  "Items": [
    {
      "Description": "Arroz Con Pollo",
      "Quantity": "1.00",
      "Unit Price": "42.00",
      "Total Price": "42.00"
    },
    {
      "Description": "(Combo) CEV MIX + CHICHA PES",
      "Quantity": "1.00",
      "Unit Price": "47.00",
      "Total Price": "47.00"
    },
    {
      "Description": "Chicha - Vaso",
      "Quantity": "1.00",
      "Unit Price": "8.00",
      "Total Price": "8.00"
    },
    {
      "Description": "Maracuya - Vaso",
      "Quantity": "1.00",
      "Unit Price": "8.00",
      "Total Price": "8.00"
    },
    {
      